# World session 1 · Who is who

**Goal:** find the people who were entered twice. In course 1 every record had an offender number, so "who is this?" was never the question. In the expanded world, 448,877 people came from several systems, and some of them were typed in a second time under a new id.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Your answers are checked against the world's **answer keys**: the generator's own record of every mess it made. A result you hand in (rows you found) is graded by **precision** (how many of yours are right) and **recall** (how many of the right ones you found).
- Nothing here can change the lake. You read the messy files as a source system delivered them.

> ⚠️ **Mock data only.** Every person, place and record in this world is invented.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and opens the world's messy files.
SESSION = "1"

import sys
from pathlib import Path

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "lakehouse" / "sessions" / "check.py").exists():
        sys.path.insert(0, str(folder))
        break

import re
import pandas as pd
from lakehouse.sessions import check
from lakehouse.sessions.check import ___

con = check.world()       # DuckDB: one view messy_<table> per messy CSV folder, every column as text
print(f"World session {SESSION} ready. Messy tables:", ", ".join(r[0] for r in con.sql("SHOW TABLES").fetchall()))

## 1. The people file
One row per person: villains (today's offenders, with a SID), heroes, secondary heroes and civilians. Look at a few, and at who has a SID.

In [ ]:
con.sql("SELECT * FROM messy_person USING SAMPLE 8 ROWS (reservoir, 42)").df()

In [ ]:
con.sql("SELECT role, count(*) AS people, count(sid_number) AS with_sid FROM messy_person GROUP BY role ORDER BY people DESC").df()

Roles are spelled several ways (that's session 1 of course 1 all over again), and only villains carry a SID. A SID is a person's state ID: two rows with the same SID are the same person.

## 2. The easy matches: the same SID twice
Group by SID and look for groups of more than one row. Within a group, call the row with the lowest `person_id` the original and the others duplicates.

In [ ]:
con.sql("""
    SELECT sid_number, list(person_id ORDER BY CAST(person_id AS BIGINT)) AS rows_with_it
    FROM messy_person WHERE sid_number IS NOT NULL
    GROUP BY sid_number HAVING count(*) > 1 LIMIT 5
""").df()

🧪 **Your turn #1.** How many rows are duplicates by SID alone (rows whose SID already belongs to a row with a lower `person_id`)?

In [ ]:
answer_1 = ___      # count the rows after the first in each SID group
check.w1_ex1(answer_1)

That's the easy part, and it's small: most people never had a SID. Heroes and civilians who were entered twice have to be found by **name and birth date**, which are messy.

## 3. Names that aren't names
Before matching on a name, look at the strangest ones.

In [ ]:
con.sql("""
    SELECT last_name, count(*) AS n FROM messy_person
    WHERE NOT regexp_matches(coalesce(last_name, 'x'), '^[A-Za-z]')
    GROUP BY last_name ORDER BY n DESC
""").df()

`N/A`, `UNKNOWN`, `UNK`, `?`, `NULL`, `NONE` and blank cells all mean *we don't know*. Matching on them would make thousands of strangers the same person: everyone named `UNK` born the same day.

**Careful with blanks.** Some cells hold only spaces and some only a tab. DuckDB's `trim(s)` removes spaces but not tabs; `trim(s, ' \t')` removes both. Run the query above and look at the rows that seem empty.

🧪 **Your turn #2.** How many last names are such stand-ins: trimmed and in capitals, one of `''`, `N/A`, `UNKNOWN`, `UNK`, `?`, `NULL` or `NONE`, once spaces **and tabs** are trimmed? (A truly empty cell is read as null, and doesn't count.)

In [ ]:
answer_2 = ___      # count the last_name values in that list
check.w1_ex2(answer_2)

## 4. Clean before you match
Two helpers. `clean_text()` turns the stand-ins into null and puts the rest in capitals (so `Adams` and `ADAMS` match). `read_date()` reads a birth date in any of the formats this file uses; world session 2 is about building one like it.

In [ ]:
con.execute("""
CREATE OR REPLACE MACRO clean_text(s) AS
    CASE WHEN upper(trim(s, ' 	')) IN ('', 'N/A', 'UNKNOWN', 'UNK', '?', 'NULL', 'NONE') THEN NULL ELSE upper(trim(s, ' 	')) END
""")
con.execute("""
CREATE OR REPLACE MACRO read_date(s) AS CASE
    WHEN regexp_matches(trim(s), '^\\d{1,2}/\\d{1,2}/\\d{2}$') THEN          -- 3/22/01: a two-digit year
        make_date(CASE WHEN CAST(split_part(trim(s), '/', 3) AS INT) <= 26 THEN 2000 ELSE 1900 END
                      + CAST(split_part(trim(s), '/', 3) AS INT),
                  CAST(split_part(trim(s), '/', 1) AS INT), CAST(split_part(trim(s), '/', 2) AS INT))
    ELSE COALESCE(try_strptime(trim(s), '%Y-%m-%d'),          -- 2001-03-22
                  try_strptime(trim(s), '%Y-%m-%d %H:%M:%S'), -- 2001-03-22 00:00:00
                  try_strptime(trim(s), '%m/%d/%Y'),          -- 03/22/2001
                  try_strptime(trim(s), '%m-%d-%Y'),          -- 03-22-2001
                  try_strptime(trim(s), '%Y%m%d'),            -- 20010322
                  try_strptime(trim(s), '%d-%b-%Y'),          -- 22-MAR-2001
                  try_strptime(trim(s), '%b %d %Y'),          -- Mar 22 2001
                  try_strptime(trim(s), '%B %d, %Y'))::DATE   -- March 22, 2001
END
""")
con.sql("SELECT clean_text('  Adams '), clean_text('UNK'), read_date('3/22/01'), read_date('March 22, 2001')").fetchall()

## 5. A match key
One key per person: their SID when there is one, otherwise cleaned last name, first name and birth date together. A key with any part unknown is null and matches nothing: better to miss a duplicate than to merge two strangers.

In [ ]:
con.sql("""
    SELECT person_id, COALESCE('SID ' || sid_number, clean_text(last_name) || '|' || clean_text(first_name) || '|' || read_date(birth_date)) AS match_key
    FROM messy_person USING SAMPLE 6 ROWS (reservoir, 11)
""").df()

🧪 **Your turn #3.** Build `duplicates`: one row per duplicate, with columns `dup_id` (the duplicate's `person_id`) and `original_id` (the lowest `person_id` sharing its match key). The bar is 99% precision and 90% recall.

In [ ]:
duplicates = ___      # a DataFrame with dup_id and original_id
check.w1_ex3(duplicates)

## 6. How the lake does it
The lake's silver layer resolves people the same way and keeps every row, giving each a `person_key`: the original's id. Nothing is deleted; links still work. Gold has one row per `person_key`.

In [ ]:
from lakehouse import lake
silver = lake.lake_ro_connect()
silver.sql("SELECT count(*) AS rows, count(DISTINCT person_key) AS people FROM lake.world_silver.person").df()

## Wrap-up
- A shared identifier (SID) finds duplicates exactly, but only for the people who have one.
- For everyone else, clean first (stand-ins to null, one case, one date format), then match on several fields together.
- A key with an unknown part matches nothing. Precision first: merging two people is worse than missing one duplicate.
- Keep every row and record who it really is (`person_key`), instead of deleting duplicates.

## Answers
Try each exercise first. Each answer replaces the `___` line in its 🧪 cell.

**#1** Number the rows in each SID group by `person_id`; everything after the first is a duplicate.

```python
answer_1 = con.sql("""
    SELECT count(*) FROM (
        SELECT row_number() OVER (PARTITION BY sid_number ORDER BY CAST(person_id AS BIGINT)) AS n
        FROM messy_person WHERE sid_number IS NOT NULL
    ) WHERE n > 1
""").fetchone()[0]
```

**#2** Trim spaces and tabs, capitalize, and count the stand-ins.

```python
answer_2 = con.sql("""
    SELECT count(*) FROM messy_person
    WHERE upper(trim(last_name, ' \t')) IN ('', 'N/A', 'UNKNOWN', 'UNK', '?', 'NULL', 'NONE')
""").fetchone()[0]
```

**#3** SID first, else the cleaned name and birth date; the lowest id in each key is the original.

```python
duplicates = con.sql("""
    WITH people AS (
        SELECT CAST(person_id AS BIGINT) AS person_id,
               COALESCE('SID ' || nullif(trim(sid_number), ''),
                        clean_text(last_name) || '|' || clean_text(first_name) || '|' || read_date(birth_date)) AS match_key
        FROM messy_person
    )
    SELECT person_id AS dup_id, min(person_id) OVER (PARTITION BY match_key) AS original_id
    FROM people
    WHERE match_key IS NOT NULL
    QUALIFY dup_id <> original_id
""").df()
```